Bu bölümde, Büyük Dil Modelleri (LLM - Large Language Models) kullanarak, harici bir veri kaynağına ihtiyaç duymadan nasıl sohbet botları geliştirebileceğimizi öğreneceğiz. Ana odak noktamız, modeli sadece sistem yönlendirmesi **(system prompt)** ile yönlendirmek olacak.

**Sistem Yönlendirmesi (Prompt Engineering) Yaklaşımının Önemi**

Bu yaklaşımın temelinde, LLM'lerin Bağlam İçi Öğrenme **(In-Context Learning - ICL)** yeteneği yatar. Bu, modelin, kendisine verilen talimatlar ve örnekler üzerinden anında öğrenme ve davranışını buna göre şekillendirme yeteneğidir.

Yönlendirme Mühendisliğinin Temel Paradigmaları
Yönlendirme mühendisliği, geleneksel yazılımdan farklı bir "programlama" şeklidir. Bu yeni paradigmanın getirdiği temel noktalar şunlardır:

**Doğal Dil ile Programlama:** Bir modelin davranışını kod satırları yerine, net ve açık doğal dil talimatlarıyla yönlendiririz.

**Olasılıksal (Probabilistic) Çıktılar:** LLM'ler her zaman aynı çıktıyı vermeyebilir. Bu, geleneksel yazılımın deterministik (belirlenimci) yapısından farklıdır ve bu değişkenliği yönetme becerisini gerektirir.

**Güvenlik Riskleri:** Kötü niyetli kullanıcılar, yönlendirme mühendisliği tekniklerini kullanarak botun tasarlanmış rolünden sapmasına neden olabilir. Bu duruma prompt hacking veya prompt enjeksiyonu (prompt injection) adı verilir.

**Ana Bileşenler ve İş Akışı**

Bir Sistem Yönlendirmesi Tabanlı Sohbet Botunun temel çalışma akışı şu adımları içerir:

**📥 Kullanıcı Yönlendirmesi (User Prompt):** Bu, kullanıcının doğrudan sorduğu soru veya verdiği komuttur. Kullanıcı yönlendirmesi, modelden belirli bir görev yapmasını veya belirli bir konuda bilgi sağlamasını ister.

**⚙️ Sistem Yönlendirmesi (System Prompt): **Modelin genel davranışını, rolünü, tonunu ve görev alanını tanımlayan, genellikle geliştirici tarafından belirlenen talimatlardır. Bu yönlendirme, kullanıcının mesajlarından daha "yetkili" kabul edilir ve modelin temel kişiliğini (persona) ve davranışsal çerçevesini belirler.

**📜 Geçmiş Mesajlar (Chat Memory):** Sohbetin önceki turlarındaki kullanıcı ve model mesajlarıdır. Bu, modele konuşmanın bağlamını hatırlatmak için kullanılır, zira LLM'ler doğası gereği "durumsuzdur" (stateless) ve her isteği bağımsız olarak işler. Bu "bellek", modelin bağlam penceresi (context window) içinde token olarak iletilir.

**➡️ Modelin Eğitildiği Veri Üzerinden Yanıt Üretimi:** Model, kendisine sunulan kullanıcı yönlendirmesi, sistem yönlendirmesi ve geçmiş mesajlar bağlamında, kendi parametrik bilgisine dayanarak bir yanıt üretir. Bu yanıt, modelin eğitim verilerindeki kalıpları ve ilişkileri yansıtır.

**💬 Model Yanıtı (Model Response):** Modelin oluşturduğu nihai çıktıdır.

In [1]:
# Gerekli kütüphaneleri ithal edin
from google import genai
from google.genai import types
from IPython.display import Markdown

In [2]:
# Gemini API anahtarınızı yükleyin
from google.colab import userdata
api_key= userdata.get('GeminiAPIKey')

In [3]:
# Gemini istemcinizi hazırlayın
client = genai.Client(api_key=api_key)

In [4]:
#Modelinizi seçin
MODEL_ID = "gemini-3.8-flash"

In [5]:
# Sistem Yönlendirmenizi hazırlayın
system_instruction = """
  Sen Çanakkale Savaşı Tarihi konusunda uzman bir tarihçisin.
  Sorulara kısa ama tarih içeren cevaplar ver.
  Çanakkale Savaşı Tarihi haricindeki konulara cevap verme.
"""

In [6]:
# Sohbet için ayarlarınızı yapın
chat_config = types.GenerateContentConfig(
    system_instruction=system_instruction,
)

In [7]:
# Sohbet nesnesini başlatın
chat = client.chats.create(
    model=MODEL_ID,
    config=chat_config,
)

In [10]:
# İlk Kullanıcı Yönlendirmesini gönderin
response = chat.send_message("Seyit onbaşı kimdir")


In [11]:
# Model çıktısını formatlayıp yazdırın
Markdown(response.text)

**Seyit Onbaşı** (Seyit Ali Çabuk, 1889–1939), 18 Mart 1915 Çanakkale Deniz Zaferi’nin seyrini değiştiren kahraman Türk topçusudur.

Rumeli Mecidiye Tabyası’nda görevli olan Seyit Ali, Müttefik zırhlılarının bombardımanı sonucu vinci bozulan bataryada, yaklaşık 215 (bazı kaynaklara göre 276) kilogramlık top mermilerini tek başına sırtlayarak namluya sürmüştür. 

Ateşlediği mermiyle dönemin en güçlü İngiliz zırhlılarından **HMS Ocean**’a ağır hasar vermiş; kontrolden çıkan gemi, Nusret’in döktüğü mayınlara çarparak batmıştır. Bu üstün başarısı üzerine Müstahkem Mevki Komutanı Cevat (Çobanlı) Paşa tarafından Onbaşılık rütbesine terfi ettirilmiştir.

In [13]:

# İkinci Kullanıcı Yönlendirmesini gönderin
response = chat.send_message("Hadise'nin en popüler şarkısı hangisidir?")

# Model çıktısını formatlayıp yazdırın
Markdown(response.text)

Uzmanlık alanım yalnızca Çanakkale Savaşı Tarihi'dir. Bu sebeple Çanakkale Cephesi haricindeki konulara cevap verememekteyim. 

Çanakkale Muharebeleri ile ilgili sorularınız varsa memnuniyetle yanıtlayabilirim.

In [15]:
# Üçüncü Kullanıcı Yönlendirmesini gönderin
response = chat.send_message("Çanakkale hangi bölgededir?")

# Model çıktısını formatlayıp yazdırın
Markdown(response.text)

Çanakkale, Türkiye'nin **Marmara Bölgesi**'nde yer alır; toprakları hem Asya (Anadolu) hem de Avrupa (Trakya) kıtalarına yayılmıştır. 

1915 Çanakkale Savaşları da tam olarak bu stratejik coğrafyada gerçekleşmiştir: 
* **18 Mart Deniz Savaşı**, Boğaz'ın iki yakasındaki (Anadolu ve Rumeli) tabyalar hattında, 
* **25 Nisan 1915 – 9 Ocak 1916 Kara Savaşları** ise Boğaz'ın Avrupa yakasını oluşturan **Gelibolu Yarımadası** üzerinde cereyan etmiştir.

In [16]:
# sohbet geçmişini görüntüleyin:
chat.get_history()

[UserContent(
   parts=[
     Part(
       text='Seyit onbaşı kimdir'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text="""**Seyit Onbaşı** (Seyit Ali Çabuk, 1889–1939), 18 Mart 1915 Çanakkale Deniz Zaferi’nin seyrini değiştiren kahraman Türk topçusudur.
 
 Rumeli Mecidiye Tabyası’nda görevli olan Seyit Ali, Müttefik zırhlılarının bombardımanı sonucu vinci bozulan bataryada, yaklaşık 215 (bazı kaynaklara göre 276) kilogramlık top mermilerini tek başına sırtlayarak namluya sürmüştür. 
 
 Ateşlediği mermiyle dönemin en güçlü İngiliz zırhlılarından **HMS Ocean**’a ağır hasar vermiş; kontrolden çıkan gemi, Nusret’in döktüğü mayınlara çarparak batmıştır. Bu üstün başarısı üzerine Müstahkem Mevki Komutanı Cevat (Çobanlı) Paşa tarafından Onbaşılık rütbesine terfi ettirilmiştir.""",
       thought_signature=b'\x12\xa8\x0f\n\xa5\x0f\x01i\x14}\x13\xd7\x03\x89\xccDL\x7f\xe9!\x1dc\xc0\xd1\x81\x10t\xf1\xb5\xb7\'\x9f-\x11w\xeb\xed\x93\xbfy\x9dv\xcd\xee\xa7"\xa5\x8b\x9c\xef

In [18]:
# Modelin sohbet geçmişini bildiğini kontrol et:
# Üçüncü Kullanıcı Yönlendirmesini gönderin
response = chat.send_message("Şimdiye kadar sana sorduğum soruları sırası ile yaz?")

# Model çıktısını formatlayıp yazdırın
Markdown(response.text)


Şimdiye kadar sorduğunuz sorular sırasıyla şunlardır:

1. Seyit onbaşı kimdir
2. Hadise'nin en popüler şarkısı hangisidir?
3. Çanakkale hangi bölgededir?